# Python Analysis - Patient cohorts, RFM, E-Rezept impact, seasonality, geography

In [ ]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

# install DuckDB so I can mix SQL with pandas when it's cleaner
!pip install duckdb -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import warnings
warnings.filterwarnings('ignore')

# chart styling - clean look without top/right borders
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize']  = 14
plt.rcParams['axes.labelsize']  = 12
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

# paths
DATA_PATH = "/content/drive/MyDrive/PharmaFlow/pharmaflow_data"
OUT_PATH  = "/content/drive/MyDrive/PharmaFlow/python_analysis_outputs"

# make sure the output folders exist
import os
os.makedirs(OUT_PATH, exist_ok=True)
os.makedirs(f"{OUT_PATH}/charts", exist_ok=True)
os.makedirs(f"{OUT_PATH}/tableau_data", exist_ok=True)

print("Setup complete")
print(f"Outputs will save to: {OUT_PATH}")


# Load data into memory and DuckDB

In [ ]:
print("Loading data...")

# load each CSV as a pandas DataFrame, parsing date columns where needed
patients     = pd.read_csv(f"{DATA_PATH}/patients.csv", parse_dates=["signup_date"])
products     = pd.read_csv(f"{DATA_PATH}/products.csv")
orders       = pd.read_csv(f"{DATA_PATH}/orders.csv", parse_dates=["order_date"])
order_items  = pd.read_csv(f"{DATA_PATH}/order_items.csv")
prescriptions= pd.read_csv(f"{DATA_PATH}/prescriptions.csv", parse_dates=["prescription_date"])
shipments    = pd.read_csv(f"{DATA_PATH}/shipments.csv")
returns      = pd.read_csv(f"{DATA_PATH}/returns.csv", parse_dates=["return_date"])
fulfillment_centers = pd.read_csv(f"{DATA_PATH}/fulfillment_centers.csv")
insurance    = pd.read_csv(f"{DATA_PATH}/insurance.csv")

# also register each DataFrame as a DuckDB table so I can use SQL when it's easier
con = duckdb.connect(database=':memory:')
for name, df in [
    ("patients", patients), ("products", products), ("orders", orders),
    ("order_items", order_items), ("prescriptions", prescriptions),
    ("shipments", shipments), ("returns", returns),
    ("fulfillment_centers", fulfillment_centers), ("insurance", insurance)
]:
    con.register(name, df)

print(f"\nLoaded {len(patients):,} patients")
print(f"Loaded {len(orders):,} orders")
print(f"Loaded {len(order_items):,} order items")
print(f"Loaded {len(shipments):,} shipments")
print(f"Loaded {len(returns):,} returns")


# Build a master analysis table

Joining everything once so I don't have to repeat the joins later.

In [ ]:
# one big joined view with all the columns I'll need downstream
master = con.execute("""
    SELECT
        o.order_id,
        o.patient_id,
        o.order_date,
        o.channel,
        o.fc_id,
        o.is_prescription_order,
        o.uses_e_rezept,
        o.order_status,
        oi.product_id,
        oi.quantity,
        oi.line_total_eur,
        p.category,
        p.requires_cold_chain,
        pat.country,
        pat.city,
        pat.age,
        pat.gender,
        pat.has_chronic_condition,
        pat.insurance_id,
        fc.fc_name,
        s.carrier,
        s.delivery_days,
        s.processing_hours,
        s.on_time_delivery,
        s.cold_chain_required
    FROM orders o
    JOIN order_items oi          ON o.order_id = oi.order_id
    JOIN products p              ON oi.product_id = p.product_id
    JOIN patients pat            ON o.patient_id = pat.patient_id
    JOIN fulfillment_centers fc  ON o.fc_id = fc.fc_id
    LEFT JOIN shipments s        ON o.order_id = s.order_id
    WHERE o.order_status = 'Delivered'
""").fetchdf()

print(f"Master table built: {len(master):,} rows x {len(master.columns)} columns")
print(f"   Columns: {list(master.columns)}")

# save it as parquet - smaller and faster than CSV for this size
master.to_parquet(f"{OUT_PATH}/master_table.parquet")
print(f"\nMaster saved to: {OUT_PATH}/master_table.parquet")


# Analysis 1: Monthly Revenue Trend (Rx vs OTC)

In [ ]:
# group revenue by month and order type, then pivot for a stacked area chart
monthly = master.copy()
monthly["month"] = monthly["order_date"].dt.to_period("M").dt.to_timestamp()
monthly["order_type"] = monthly["is_prescription_order"].map({True: "Rx", False: "OTC"})

monthly_rev = monthly.groupby(["month", "order_type"])["line_total_eur"].sum().reset_index()
monthly_rev_pivot = monthly_rev.pivot(index="month", columns="order_type", values="line_total_eur").fillna(0)

# stacked area chart - shows both volume and the Rx/OTC mix over time
fig, ax = plt.subplots(figsize=(14, 6))
monthly_rev_pivot.plot.area(ax=ax, alpha=0.7, color=["#FF6B6B", "#4ECDC4"])
ax.set_title("Monthly Revenue: Rx vs OTC (Jan 2023 - Dec 2025)", fontsize=15, fontweight='bold')
ax.set_xlabel("Month")
ax.set_ylabel("Revenue (EUR)")
# format y-axis as millions
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'EUR {x/1e6:.1f}M'))
ax.legend(title="Order Type", loc='upper left')
plt.tight_layout()
plt.savefig(f"{OUT_PATH}/charts/01_monthly_revenue_trend.png", dpi=150, bbox_inches='tight')
plt.show()

# save data for Tableau
monthly_rev.to_csv(f"{OUT_PATH}/tableau_data/01_monthly_revenue.csv", index=False)
print(f"\nSaved chart and CSV")
print(f"\nPeak month: {monthly_rev.groupby('month')['line_total_eur'].sum().idxmax().strftime('%B %Y')}")
print(f"Total revenue: EUR {monthly_rev['line_total_eur'].sum()/1e6:.1f}M")


# Analysis 2: E-Rezept impact

Did the digital prescription rollout actually speed things up?

In [ ]:
# compare processing time for Rx orders: E-Rezept vs paper
rx_orders = master[master["is_prescription_order"] == True].copy()
rx_orders["prescription_type"] = rx_orders["uses_e_rezept"].map(
    {True: "E-Rezept (Digital)", False: "Paper Prescription"}
)

# run a t-test to check if the difference is statistically significant
from scipy import stats
e_rezept_times = rx_orders[rx_orders["uses_e_rezept"]]["processing_hours"].dropna()
paper_times    = rx_orders[~rx_orders["uses_e_rezept"]]["processing_hours"].dropna()

t_stat, p_value = stats.ttest_ind(e_rezept_times, paper_times)
mean_diff = paper_times.mean() - e_rezept_times.mean()

# two charts side by side
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# left: boxplot of processing time (sampling 50k for speed)
sns.boxplot(data=rx_orders.sample(n=min(50000, len(rx_orders)), random_state=42),
            x="prescription_type", y="processing_hours", ax=axes[0],
            palette=["#FF6B6B", "#4ECDC4"])
axes[0].set_title("Processing Time: E-Rezept vs Paper", fontweight='bold')
axes[0].set_ylabel("Processing Hours")
axes[0].set_xlabel("")
axes[0].set_ylim(0, 30)

# right: quarterly adoption curve
adoption = con.execute("""
    SELECT
        DATE_TRUNC('quarter', o.order_date) AS quarter,
        ROUND(SUM(CASE WHEN o.uses_e_rezept THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS adoption_pct
    FROM orders o
    WHERE o.is_prescription_order = TRUE
    GROUP BY quarter
    ORDER BY quarter
""").fetchdf()

axes[1].plot(adoption["quarter"], adoption["adoption_pct"], marker='o', linewidth=3, color="#2E86AB")
axes[1].fill_between(adoption["quarter"], adoption["adoption_pct"], alpha=0.2, color="#2E86AB")
axes[1].set_title("E-Rezept Adoption Rate by Quarter", fontweight='bold')
axes[1].set_ylabel("Adoption %")
axes[1].set_xlabel("Quarter")
# add a halfway marker
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.savefig(f"{OUT_PATH}/charts/02_e_rezept_impact.png", dpi=150, bbox_inches='tight')
plt.show()

adoption.to_csv(f"{OUT_PATH}/tableau_data/02_e_rezept_adoption.csv", index=False)

print(f"\nE-Rezept Impact:")
print(f"   E-Rezept avg processing: {e_rezept_times.mean():.1f} hours")
print(f"   Paper avg processing:    {paper_times.mean():.1f} hours")
print(f"   Time saved:              {mean_diff:.1f} hours ({mean_diff/paper_times.mean()*100:.1f}% faster)")
print(f"   t-stat={t_stat:.2f}, p-value={p_value:.6f}")
print(f"   Adoption went from {adoption.iloc[0]['adoption_pct']:.1f}% (Q1 2023) to {adoption.iloc[-1]['adoption_pct']:.1f}% (Q4 2025)")


# Analysis 3: RFM patient segmentation

Who are the VIPs, who is churning?

In [ ]:
# RFM = Recency (days since last order), Frequency (# orders), Monetary (total spent)

# snapshot date = day after the most recent order in the data
snapshot_date = master["order_date"].max() + pd.Timedelta(days=1)

# calculate R, F, M for each patient
rfm = master.groupby("patient_id").agg(
    recency=("order_date", lambda x: (snapshot_date - x.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("line_total_eur", "sum")
).reset_index()

# score each metric 1-5 using quintiles (5 = best)
# recency is inverse - lower days = better score
rfm["R_score"] = pd.qcut(rfm["recency"], 5, labels=[5,4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
rfm["RFM_score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

# bucket into business-friendly segments
def segment(row):
    if row["RFM_score"] >= 13: return "Champions"
    if row["RFM_score"] >= 10: return "Loyal"
    if row["RFM_score"] >= 8:  return "Potential Loyalists"
    if row["R_score"]   <= 2:  return "At Risk"
    if row["R_score"] == 1 and row["F_score"] <= 2: return "Churned"
    return "Regular"

rfm["segment"] = rfm.apply(segment, axis=1)

# aggregate by segment
seg_summary = rfm.groupby("segment").agg(
    patients=("patient_id", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum")
).reset_index().sort_values("total_revenue", ascending=False)

# two bar charts: patient count vs revenue per segment
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = ["#06A77D", "#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#6C757D"]

# left: patient count
axes[0].barh(seg_summary["segment"], seg_summary["patients"], color=colors[:len(seg_summary)])
axes[0].set_title("Patients per Segment", fontweight='bold')
axes[0].set_xlabel("Number of Patients")
for i, v in enumerate(seg_summary["patients"]):
    axes[0].text(v, i, f' {v:,}', va='center')

# right: revenue
axes[1].barh(seg_summary["segment"], seg_summary["total_revenue"]/1e6, color=colors[:len(seg_summary)])
axes[1].set_title("Revenue per Segment (EUR M)", fontweight='bold')
axes[1].set_xlabel("Revenue (EUR M)")
for i, v in enumerate(seg_summary["total_revenue"]/1e6):
    axes[1].text(v, i, f' EUR {v:.1f}M', va='center')

plt.tight_layout()
plt.savefig(f"{OUT_PATH}/charts/03_rfm_segmentation.png", dpi=150, bbox_inches='tight')
plt.show()

rfm.to_csv(f"{OUT_PATH}/tableau_data/03_rfm_segments.csv", index=False)
seg_summary.to_csv(f"{OUT_PATH}/tableau_data/03_segment_summary.csv", index=False)

print("\nRFM Segment Summary:")
print(seg_summary.to_string(index=False))

# how much of revenue comes from the top segments
top_segments_pct = seg_summary[seg_summary["segment"].isin(["Champions", "Loyal"])]["total_revenue"].sum() / seg_summary["total_revenue"].sum() * 100
print(f"\nChampions + Loyal = {top_segments_pct:.1f}% of total revenue")


# Analysis 4: Geographic revenue analysis

Where should marketing focus?

In [ ]:
# city-level revenue breakdown
geo = con.execute("""
    SELECT
        country,
        city,
        COUNT(DISTINCT patient_id) AS patients,
        COUNT(DISTINCT order_id) AS orders,
        ROUND(SUM(line_total_eur), 2) AS revenue_eur,
        ROUND(SUM(line_total_eur) / COUNT(DISTINCT patient_id), 2) AS revenue_per_patient
    FROM master
    GROUP BY country, city
    ORDER BY revenue_eur DESC
""").fetchdf()

# country-level rollup
country_summary = geo.groupby("country").agg(
    cities=("city", "nunique"),
    patients=("patients", "sum"),
    orders=("orders", "sum"),
    revenue_eur=("revenue_eur", "sum")
).reset_index().sort_values("revenue_eur", ascending=False)

# two charts: country revenue + top 15 cities
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# country bars
axes[0].bar(country_summary["country"], country_summary["revenue_eur"]/1e6,
            color=["#FFCE00", "#ED2939", "#DC143C", "#FF6B35"])
axes[0].set_title("Revenue by Country", fontweight='bold')
axes[0].set_ylabel("Revenue (EUR M)")
axes[0].set_xlabel("Country")
for i, v in enumerate(country_summary["revenue_eur"]/1e6):
    axes[0].text(i, v, f'EUR {v:.1f}M', ha='center', va='bottom', fontweight='bold')

# top 15 cities
top_cities = geo.head(15)
axes[1].barh(top_cities["city"][::-1], top_cities["revenue_eur"][::-1]/1e6, color="#2E86AB")
axes[1].set_title("Top 15 Cities by Revenue", fontweight='bold')
axes[1].set_xlabel("Revenue (EUR M)")

plt.tight_layout()
plt.savefig(f"{OUT_PATH}/charts/04_geographic_revenue.png", dpi=150, bbox_inches='tight')
plt.show()

geo.to_csv(f"{OUT_PATH}/tableau_data/04_geo_revenue.csv", index=False)
country_summary.to_csv(f"{OUT_PATH}/tableau_data/04_country_summary.csv", index=False)

print("\nCountry Performance:")
print(country_summary.to_string(index=False))
print(f"\nTop city: {geo.iloc[0]['city']} ({geo.iloc[0]['country']}) - EUR {geo.iloc[0]['revenue_eur']/1e6:.1f}M")


# Analysis 5: Operations dashboard data

Where are the operational risks?

In [ ]:
# FC performance with all the key metrics
fc_perf = con.execute("""
    SELECT
        fc.fc_name,
        fc.country,
        COUNT(DISTINCT s.shipment_id) AS shipments,
        ROUND(AVG(s.delivery_days), 2) AS avg_delivery_days,
        ROUND(AVG(s.processing_hours), 2) AS avg_processing_hours,
        ROUND(SUM(CASE WHEN s.on_time_delivery THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS otif_pct
    FROM shipments s
    JOIN orders o ON s.order_id = o.order_id
    JOIN fulfillment_centers fc ON o.fc_id = fc.fc_id
    GROUP BY fc.fc_name, fc.country
    ORDER BY otif_pct DESC
""").fetchdf()

# carrier performance
carrier_perf = con.execute("""
    SELECT
        carrier,
        COUNT(*) AS shipments,
        ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
        ROUND(SUM(CASE WHEN on_time_delivery THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS otif_pct
    FROM shipments
    GROUP BY carrier
    ORDER BY otif_pct DESC
""").fetchdf()

# two horizontal bar charts with a target line at 80%
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# FC chart
axes[0].barh(fc_perf["fc_name"][::-1], fc_perf["otif_pct"][::-1], color="#06A77D")
axes[0].axvline(x=80, color='red', linestyle='--', alpha=0.5, label='Target 80%')
axes[0].set_title("OTIF % by Fulfillment Center", fontweight='bold')
axes[0].set_xlabel("On-Time-In-Full %")
axes[0].set_xlim(60, 100)
axes[0].legend()
for i, v in enumerate(fc_perf["otif_pct"][::-1]):
    axes[0].text(v, i, f' {v}%', va='center')

# Carrier chart with traffic-light coloring
# green if >=80, amber if 75-80, red below
colors_carrier = ["#06A77D" if x >= 80 else "#F18F01" if x >= 75 else "#C73E1D" for x in carrier_perf["otif_pct"][::-1]]
axes[1].barh(carrier_perf["carrier"][::-1], carrier_perf["otif_pct"][::-1], color=colors_carrier)
axes[1].axvline(x=80, color='red', linestyle='--', alpha=0.5, label='Target 80%')
axes[1].set_title("OTIF % by Carrier", fontweight='bold')
axes[1].set_xlabel("On-Time-In-Full %")
axes[1].set_xlim(60, 100)
axes[1].legend()
for i, v in enumerate(carrier_perf["otif_pct"][::-1]):
    axes[1].text(v, i, f' {v}%', va='center')

plt.tight_layout()
plt.savefig(f"{OUT_PATH}/charts/05_operations_otif.png", dpi=150, bbox_inches='tight')
plt.show()

fc_perf.to_csv(f"{OUT_PATH}/tableau_data/05_fc_performance.csv", index=False)
carrier_perf.to_csv(f"{OUT_PATH}/tableau_data/05_carrier_performance.csv", index=False)

print("\nCarrier OTIF Performance:")
print(carrier_perf.to_string(index=False))
print(f"\nBest: {carrier_perf.iloc[0]['carrier']} ({carrier_perf.iloc[0]['otif_pct']}%)")
print(f"Worst: {carrier_perf.iloc[-1]['carrier']} ({carrier_perf.iloc[-1]['otif_pct']}%)")
print(f"Gap: {carrier_perf.iloc[0]['otif_pct'] - carrier_perf.iloc[-1]['otif_pct']:.1f} points")


# Analysis 6: Headline KPIs for the dashboard

In [ ]:
# pull together all the top-line numbers in one place
kpis = {}

# revenue & orders
kpis["total_revenue_eur"] = master["line_total_eur"].sum()
kpis["total_orders"]      = master["order_id"].nunique()
kpis["total_patients"]    = master["patient_id"].nunique()
kpis["aov_eur"]           = kpis["total_revenue_eur"] / kpis["total_orders"]

# Rx vs OTC
rx_revenue = master[master["is_prescription_order"]]["line_total_eur"].sum()
otc_revenue = master[~master["is_prescription_order"]]["line_total_eur"].sum()
kpis["rx_revenue_share_pct"] = round(rx_revenue / (rx_revenue + otc_revenue) * 100, 2)

# E-Rezept adoption in 2025 only - shows current state
rx_2025 = master[(master["is_prescription_order"]) & (master["order_date"].dt.year == 2025)]
kpis["e_rezept_adoption_2025_pct"] = round(rx_2025["uses_e_rezept"].mean() * 100, 2)

# operations
kpis["overall_otif_pct"] = round(shipments["on_time_delivery"].mean() * 100, 2)
kpis["cold_chain_otif_pct"] = round(shipments[shipments["cold_chain_required"]]["on_time_delivery"].mean() * 100, 2)

# returns
kpis["return_rate_pct"] = round(len(returns) / kpis["total_orders"] * 100, 2)

# print everything
print("="*60)
print("PHARMAFLOW - HEADLINE KPIs")
print("="*60)
print(f"  Total Revenue:         EUR {kpis['total_revenue_eur']/1e6:.1f}M")
print(f"  Total Orders:          {kpis['total_orders']:,}")
print(f"  Total Patients:        {kpis['total_patients']:,}")
print(f"  Avg Order Value:       EUR {kpis['aov_eur']:.2f}")
print(f"  Rx Revenue Share:      {kpis['rx_revenue_share_pct']}%")
print(f"  E-Rezept Adoption '25: {kpis['e_rezept_adoption_2025_pct']}%")
print(f"  Overall OTIF:          {kpis['overall_otif_pct']}%")
print(f"  Cold Chain OTIF:       {kpis['cold_chain_otif_pct']}%")
print(f"  Return Rate:           {kpis['return_rate_pct']}%")
print("="*60)

# save as a CSV so I can use it directly in Tableau
kpi_df = pd.DataFrame([
    {"KPI": "Total Revenue (EUR M)",   "Value": round(kpis["total_revenue_eur"]/1e6, 1)},
    {"KPI": "Total Orders",             "Value": kpis["total_orders"]},
    {"KPI": "Total Patients",           "Value": kpis["total_patients"]},
    {"KPI": "AOV (EUR)",                "Value": round(kpis["aov_eur"], 2)},
    {"KPI": "Rx Revenue Share (%)",     "Value": kpis["rx_revenue_share_pct"]},
    {"KPI": "E-Rezept Adoption 2025 (%)", "Value": kpis["e_rezept_adoption_2025_pct"]},
    {"KPI": "Overall OTIF (%)",         "Value": kpis["overall_otif_pct"]},
    {"KPI": "Cold Chain OTIF (%)",      "Value": kpis["cold_chain_otif_pct"]},
    {"KPI": "Return Rate (%)",          "Value": kpis["return_rate_pct"]},
])
kpi_df.to_csv(f"{OUT_PATH}/tableau_data/00_headline_kpis.csv", index=False)
print("\nKPIs saved")


# Final check - verify all outputs are ready for Tableau

In [ ]:
import os

print("Files ready for Tableau dashboard:\n")
tableau_files = sorted(os.listdir(f"{OUT_PATH}/tableau_data"))
for f in tableau_files:
    size_kb = os.path.getsize(f"{OUT_PATH}/tableau_data/{f}") / 1024
    print(f"   {f}  ({size_kb:.1f} KB)")

print(f"\nCharts saved:\n")
chart_files = sorted(os.listdir(f"{OUT_PATH}/charts"))
for f in chart_files:
    size_kb = os.path.getsize(f"{OUT_PATH}/charts/{f}") / 1024
    print(f"   {f}  ({size_kb:.1f} KB)")


# Extra numbers for the business insights writeup

In [ ]:
# pulling a few extra numbers I'll need for the insights document

# 1. what % of revenue comes from the top 20% of patients (Pareto check)
patient_revenue = master.groupby("patient_id")["line_total_eur"].sum().sort_values(ascending=False)
total_revenue = patient_revenue.sum()
top_20_cutoff = int(len(patient_revenue) * 0.20)
top_20_revenue = patient_revenue.iloc[:top_20_cutoff].sum()
top_20_pct = round(top_20_revenue / total_revenue * 100, 1)

print(f"Top 20% of patients = {top_20_pct}% of revenue")
print(f"   ({top_20_cutoff:,} patients out of {len(patient_revenue):,})")

# 2. how much more do chronic patients spend vs acute
chronic_rev_per_patient = master[master["has_chronic_condition"]].groupby("patient_id")["line_total_eur"].sum().mean()
acute_rev_per_patient   = master[~master["has_chronic_condition"]].groupby("patient_id")["line_total_eur"].sum().mean()
chronic_multiplier = round(chronic_rev_per_patient / acute_rev_per_patient, 2)

print(f"\nChronic patient revenue/patient:   EUR {chronic_rev_per_patient:.2f}")
print(f"Acute patient revenue/patient:     EUR {acute_rev_per_patient:.2f}")
print(f"Chronic patients spend {chronic_multiplier}x more than acute")

# 3. revenue split by country
country_rev = master.groupby("country")["line_total_eur"].sum().sort_values(ascending=False)
country_pct = (country_rev / country_rev.sum() * 100).round(1)
print(f"\nRevenue by country:")
for c, pct in country_pct.items():
    print(f"   {c}: {pct}% (EUR {country_rev[c]/1e6:.1f}M)")

# 4. Medical Devices - the highest-revenue and highest-return category
md_revenue = master[master["category"] == "Medical_Devices"]["line_total_eur"].sum()
md_revenue_pct = round(md_revenue / master["line_total_eur"].sum() * 100, 1)

# return rate for Medical Devices specifically
md_orders = master[master["category"] == "Medical_Devices"]["order_id"].nunique()
md_returns = con.execute("""
    SELECT COUNT(DISTINCT r.return_id) as cnt
    FROM returns r
    JOIN order_items oi ON r.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE p.category = 'Medical_Devices'
""").fetchone()[0]
md_return_rate = round(md_returns / md_orders * 100, 2)

print(f"\nMedical Devices revenue: EUR {md_revenue/1e6:.1f}M ({md_revenue_pct}% of total)")
print(f"Medical Devices return rate: {md_return_rate}%")

# 5. carrier gap - DHL vs GLS
dhl_otif = round(shipments[shipments["carrier"] == "DHL"]["on_time_delivery"].mean() * 100, 2)
gls_otif = round(shipments[shipments["carrier"] == "GLS"]["on_time_delivery"].mean() * 100, 2)
print(f"\nDHL OTIF: {dhl_otif}%  vs  GLS OTIF: {gls_otif}%  (gap: {dhl_otif - gls_otif:.1f} pts)")

# 6. E-Rezept processing time savings
e_rezept_time = master[(master["is_prescription_order"]) & (master["uses_e_rezept"])]["processing_hours"].mean()
paper_time    = master[(master["is_prescription_order"]) & (~master["uses_e_rezept"])]["processing_hours"].mean()
time_saved_pct = round((1 - e_rezept_time/paper_time) * 100, 1)
print(f"\nE-Rezept processing: {e_rezept_time:.1f}h vs Paper: {paper_time:.1f}h")
print(f"Time saved: {time_saved_pct}%")
